# SHAP Explainability

Adding per-instance feature attribution on top of the tuned XGBoost model
from notebooks/03_model_training.ipynb. This produces the reasoning that
will populate fault_detections.shap_summary for each detected fault, and
gives the Predictive LLM agent real, grounded evidence to cite
rather than an opaque confidence score.

In [1]:
import pandas as pd
import json
import xgboost as xgb
import shap
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv("../.env")

model = xgb.XGBClassifier()
model.load_model("../models/xgboost_fault_detector.json")

with open("../models/label_encoder_classes.json") as f:
    class_names = json.load(f)

print("Model loaded. Classes:", class_names)

user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]
db = os.environ["POSTGRES_DB"]
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")

test_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'test'", engine)
feature_cols = [c for c in test_df.columns if c not in
                ("id", "ahu_id", "reading_time", "labeling", "split")]
X_test = test_df[feature_cols]

print(f"Test set: {X_test.shape}")

Model loaded. Classes: ['Normal condition', 'Return air temperature fault', 'Supply air temperature fault', 'Supply fan fault', 'Valve position fault']
Test set: (35107, 18)


## Creating the explainer

Using SHAP's TreeExplainer, which computes exact SHAP values efficiently
for tree-based models like XGBoost, rather than the slower approximation
methods needed for arbitrary model types.

In [2]:
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the test set. For multi-class models, this
# returns one array of values per class, shape (n_samples, n_features, n_classes).
shap_values = explainer.shap_values(X_test)

print("SHAP values shape:", shap_values.shape)

SHAP values shape: (35107, 18, 5)


## Explaining a single prediction

Picking one row the model flagged as a fault, and showing which features
drove that specific decision. This is the format that will eventually
populate fault_detections.shap_summary for each real detected fault.

In [5]:
preds = model.predict(X_test)
fault_class_idx = class_names.index("Supply fan fault")

fault_rows = X_test[preds == fault_class_idx]
sample_idx = fault_rows.index[0]
sample_position = X_test.index.get_loc(sample_idx)

print(f"Explaining row at position {sample_position} (predicted: {class_names[fault_class_idx]})")
print(X_test.iloc[sample_position])

Explaining row at position 101 (predicted: Supply fan fault)
set_point_temp           22.00
return_temp              25.14
supply_air_temp          25.07
supply_fan                0.00
valve_position            0.00
heating_supply_temp_1      NaN
heating_supply_temp_2      NaN
total_heating_pump         NaN
heating_pump_1             NaN
heating_pump_2             NaN
heating_pump_3             NaN
cooling_supply_temp_1    12.96
cooling_supply_temp_2    12.74
total_cooling_pump        1.00
cooling_pump_1            0.00
cooling_pump_2            0.00
cooling_pump_3            0.50
cooling_pump_4            0.50
Name: 101, dtype: float64


In [6]:
row_shap = shap_values[sample_position, :, fault_class_idx]

contributions = pd.Series(row_shap, index=feature_cols).sort_values(key=abs, ascending=False)
print("Top feature contributions toward 'Supply fan fault' prediction:")
print(contributions.head(8))

Top feature contributions toward 'Supply fan fault' prediction:
set_point_temp           1.723595
return_temp              1.385238
supply_fan               0.819378
total_cooling_pump       0.636221
cooling_supply_temp_1    0.483415
total_heating_pump      -0.455981
cooling_pump_1          -0.116734
cooling_pump_3           0.114547
dtype: float32


### Interpreting the explanation

For this row (predicted Supply fan fault), the top contributing features:

| Feature | Value | SHAP contribution |
|---|---|---|
| set_point_temp | 22.00 | +1.72 |
| return_temp | 25.14 | +1.39 |
| supply_fan | 0.00 | +0.82 |
| total_cooling_pump | 1.00 | +0.64 |
| cooling_supply_temp_1 | 12.96 | +0.48 |
| total_heating_pump | NaN | -0.46 |

set_point_temp and return_temp together capture the core signal: the room
is running about 3 degrees warmer than its target. supply_fan sits at
0.00 despite that gap, the fan isn't responding to demand. total_cooling_pump
being active alongside an idle fan reinforces this, cooling is being
called for but not delivered. total_heating_pump pushes slightly against
the prediction, correctly reflecting that the AHU is in cooling mode for
this reading, not heating.

## Explaining a Valve position fault prediction

Valve position fault is the class added via SMOTE. Explaining a real
prediction on this class specifically, to confirm SHAP produces
genuinely meaningful reasoning here too, not just on classes with
abundant real training data.

In [7]:
vpf_class_idx = class_names.index("Valve position fault")

vpf_rows = X_test[preds == vpf_class_idx]
sample_idx_vpf = vpf_rows.index[0]
sample_position_vpf = X_test.index.get_loc(sample_idx_vpf)

print(f"Explaining row at position {sample_position_vpf} (predicted: {class_names[vpf_class_idx]})")
print(X_test.iloc[sample_position_vpf])

Explaining row at position 1072 (predicted: Valve position fault)
set_point_temp           25.00
return_temp              21.96
supply_air_temp          26.18
supply_fan                1.00
valve_position            0.00
heating_supply_temp_1      NaN
heating_supply_temp_2      NaN
total_heating_pump         NaN
heating_pump_1             NaN
heating_pump_2             NaN
heating_pump_3             NaN
cooling_supply_temp_1    13.06
cooling_supply_temp_2    11.46
total_cooling_pump        2.00
cooling_pump_1            0.00
cooling_pump_2            0.00
cooling_pump_3            1.00
cooling_pump_4            1.00
Name: 1072, dtype: float64


In [8]:
row_shap_vpf = shap_values[sample_position_vpf, :, vpf_class_idx]

contributions_vpf = pd.Series(row_shap_vpf, index=feature_cols).sort_values(key=abs, ascending=False)
print("Top feature contributions toward 'Valve position fault' prediction:")
print(contributions_vpf.head(8))

Top feature contributions toward 'Valve position fault' prediction:
supply_fan               2.481580
cooling_pump_3           0.832033
valve_position           0.752634
return_temp              0.602670
set_point_temp           0.502464
cooling_supply_temp_2    0.435480
cooling_supply_temp_1   -0.271968
cooling_pump_1           0.187443
dtype: float32


### Interpreting the explanation

For this row (predicted Valve position fault), the top contributing features:

| Feature | Value | SHAP contribution |
|---|---|---|
| supply_fan | 1.00 | +2.48 |
| cooling_pump_3 | 1.00 | +0.83 |
| valve_position | 0.00 | +0.75 |
| return_temp | 21.96 | +0.60 |
| set_point_temp | 25.00 | +0.50 |
| cooling_supply_temp_2 | 11.46 | +0.44 |

The fan running at full (supply_fan = 1.00) alongside an active cooling
pump (cooling_pump_3 = 1.00) shows the system is actively trying to cool
the space. valve_position sitting at 0.00 despite that active demand is
the actual fault signature, the valve isn't opening even though cooling
is clearly being called for elsewhere in the system. supply_fan carries
the largest single contribution here, which makes sense: a valve stuck at
0% only looks anomalous in context, once fan and pump activity confirm
cooling is genuinely needed. On its own, valve_position = 0.00 is
completely normal (it's also 0.00 during Normal condition much of the
time), so the model has learned to weigh it relative to what the rest of
the system is doing, not in isolation.

## Building the shap_summary structure

Converting a row's SHAP values into the small JSON object that will be
stored in fault_detections.shap_summary, keeping only the top contributing
features rather than all 18, since only a handful meaningfully drive any
single prediction.

In [9]:
def build_shap_summary(row_position: int, predicted_class_idx: int, top_n: int = 5) -> dict:
    row_shap = shap_values[row_position, :, predicted_class_idx]
    contributions = pd.Series(row_shap, index=feature_cols).sort_values(key=abs, ascending=False)

    top_features = []
    for feature, value in contributions.head(top_n).items():
        top_features.append({
            "feature": feature,
            "shap_value": round(float(value), 4),
            "reading": None if pd.isna(X_test.iloc[row_position][feature])
                       else round(float(X_test.iloc[row_position][feature]), 2)
        })

    return {
        "predicted_class": class_names[predicted_class_idx],
        "top_features": top_features
    }


summary_vpf = build_shap_summary(sample_position_vpf, vpf_class_idx)
print(json.dumps(summary_vpf, indent=2))

{
  "predicted_class": "Valve position fault",
  "top_features": [
    {
      "feature": "supply_fan",
      "shap_value": 2.4816,
      "reading": 1.0
    },
    {
      "feature": "cooling_pump_3",
      "shap_value": 0.832,
      "reading": 1.0
    },
    {
      "feature": "valve_position",
      "shap_value": 0.7526,
      "reading": 0.0
    },
    {
      "feature": "return_temp",
      "shap_value": 0.6027,
      "reading": 21.96
    },
    {
      "feature": "set_point_temp",
      "shap_value": 0.5025,
      "reading": 25.0
    }
  ]
}


## Summary

- SHAP's TreeExplainer computes exact per-instance feature attributions
  for the trained XGBoost model, loaded fresh from
  models/xgboost_fault_detector.json rather than retrained, guaranteeing
  consistency with the model actually evaluated in
  notebooks/03_model_training.ipynb.
- Two explained predictions confirm SHAP produces coherent, physically
  sensible reasoning across different classes, not arbitrary noise: Supply
  fan fault (set_point_temp and return_temp reveal a real temperature gap
  the idle fan isn't responding to) and Valve position fault (supply_fan
  and cooling_pump activity confirm cooling is being demanded elsewhere in
  the system, making the closed valve the actual anomaly, not just a
  normal idle reading).
- The Valve position fault explanation specifically shows the model
  weighing valve_position relative to surrounding system activity rather
  than in isolation, since 0.00 alone is also the normal reading much of
  the time. This is consistent with feature importance findings from
  model training, no single sensor determines a prediction on its own.
- build_shap_summary() converts a row's SHAP values into a compact JSON
  structure (predicted class plus top 5 contributing features with their
  values and readings), matching the shape needed for
  fault_detections.shap_summary. This function is what the Predictive
  agent will call on each newly detected fault, one row at a time, not on
  batches, since SHAP computation on a single row is fast.